# Birthday Distribution Analysis

Analysis of how many FIFA players were born on each calendar day (month/day), ignoring the year.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Load data
df = pd.read_csv('../Datasets/Football Players Data/fifa_players.csv')

# Parse birth_date (format: M/D/YYYY)
df['birth_date'] = pd.to_datetime(df['birth_date'], format='%m/%d/%Y')

# Extract calendar day (month and day, ignoring year)
df['month'] = df['birth_date'].dt.month
df['day'] = df['birth_date'].dt.day
df['month_day'] = df['birth_date'].dt.strftime('%m-%d')

print(f"Total players: {len(df)}")
print(f"Date range: {df['birth_date'].min().date()} – {df['birth_date'].max().date()}")
df['birth_date'].head()


In [ ]:
# Count players born on each calendar day (MM-DD), sorted chronologically
birthday_counts = (
    df.groupby('month_day')
    .size()
    .reset_index(name='player_count')
    .sort_values('month_day')
)

print(f"Distinct calendar days with at least one player: {len(birthday_counts)}")
print("\nTop 10 most common birthdays:")
print(birthday_counts.sort_values('player_count', ascending=False).head(10).to_string(index=False))


In [ ]:
# Build a complete calendar with all 366 days (including Feb 29) for a clean x-axis
all_days = pd.date_range('2000-01-01', '2000-12-31', freq='D')  # 2000 is a leap year
all_days_df = pd.DataFrame({'month_day': all_days.strftime('%m-%d')})

# Merge so days with zero players still appear
plot_df = all_days_df.merge(birthday_counts, on='month_day', how='left').fillna(0)
plot_df['player_count'] = plot_df['player_count'].astype(int)

# Month boundary positions for vertical grid lines
month_starts = [
    i for i, md in enumerate(plot_df['month_day'])
    if md.endswith('-01')
]
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar(range(len(plot_df)), plot_df['player_count'], color='steelblue', width=1.0)

# Add vertical month separators and month labels
for pos, label in zip(month_starts, month_labels):
    ax.axvline(pos, color='grey', linewidth=0.6, linestyle='--')
    ax.text(pos + 15, ax.get_ylim()[1] * 0.95, label, fontsize=9, color='dimgrey')

ax.set_xlabel('Day of the Year (MM-DD)')
ax.set_ylabel('Number of Players')
ax.set_title('FIFA Players – Number of Players Born on Each Calendar Day')
ax.set_xticks(month_starts)
ax.set_xticklabels([plot_df['month_day'].iloc[p] for p in month_starts], rotation=45, ha='right')
ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()
